<a href="https://colab.research.google.com/github/tregrwells/TRC-V24-Tregrwells/blob/main/Copy_of_python_trc_v16_final_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================================
# TRC V16 – Final Complete Script (Fixed Syntax)
# ======================================================================

import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
import glob
import urllib.request
import re
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar

# ----------------------------------------------------------------------
# 0. Install/compile CLASS if needed
# ----------------------------------------------------------------------
if not os.path.exists("class_public/class"):
    print("CLASS not found. Cloning and compiling...")
    if not os.path.exists("class_public"):
        subprocess.run(["git", "clone", "https://github.com/lesgourg/class_public.git"], check=True)
    subprocess.run(["make", "-C", "class_public"], check=True)
    print("CLASS compiled.")
else:
    print("CLASS already compiled.")

# ----------------------------------------------------------------------
# 1. TRC V16 parameters
# ----------------------------------------------------------------------
TRC = {
    "Omega_m": 0.1090,
    "Omega_b": 0.05,
    "H0": 69.99,
    "DE_init": 0.7126,
    "eps0": 0.2435,
    "a_crit": 0.5542,
    "Delta_a": 0.01,
    "A_slab": 0.0003,
    "a_peak": 0.0008,
    "sigma_log": 0.15,
    "beta": 0.921,      # gives μ = 1+2β² = 2.6965
}

h = TRC["H0"] / 100.0
omega_b = TRC["Omega_b"] * h**2
omega_cdm = (TRC["Omega_m"] - TRC["Omega_b"]) * h**2
mu_const = 1 + 2 * TRC["beta"]**2
tau_reio = 0.0544   # Planck 2018 best-fit

# ----------------------------------------------------------------------
# 2. Background functions
# ----------------------------------------------------------------------
def epsilon(a):
    eps0 = TRC["eps0"]
    a_crit = TRC["a_crit"]
    Delta_a = TRC["Delta_a"]
    return eps0 + (1 - eps0) / (1 + np.exp(-(a - a_crit) / Delta_a))

def Omega_DE_V13(a):
    return TRC["DE_init"] + (0.8485 - TRC["DE_init"]) * epsilon(a)

def Omega_DE_slab(a):
    log_a = np.log(a)
    log_a_peak = np.log(TRC["a_peak"])
    return TRC["A_slab"] * 1e13 * np.exp(-(log_a - log_a_peak)**2 / (2 * TRC["sigma_log"]**2))

def Omega_DE_total(a):
    return Omega_DE_V13(a) + Omega_DE_slab(a)

def H_of_a(a):
    Om_m = TRC["Omega_m"]
    Om_b = TRC["Omega_b"]
    H0 = TRC["H0"]
    Omega_r = 8.4e-5
    E2 = Omega_DE_total(a) + Om_m * a**(-3) + Om_b * a**(-3) + Omega_r * a**(-4)
    return H0 * np.sqrt(E2)

# ----------------------------------------------------------------------
# 3. Generate H(z) table
# ----------------------------------------------------------------------
z_vals = np.linspace(0.0, 1100.0, 5000)
a_vals = 1.0 / (1.0 + z_vals)
H_vals = np.array([H_of_a(a) for a in a_vals])
bg_lines = [f"    {z:.6e} {H:.6e}" for z, H in zip(z_vals, H_vals)]
bg_str = "\n".join(bg_lines)

# ----------------------------------------------------------------------
# 4. Write CLASS input file
# ----------------------------------------------------------------------
content = f"""
# TRC V16 – Final (constant μ, explicit τ)
h = {h:.6f}
omega_b = {omega_b:.6f}
omega_cdm = {omega_cdm:.6f}
Omega_k = 0.0
T_cmb = 2.7255
N_ur = 3.046
tau_reio = {tau_reio:.6f}

custom_background = yes
custom_H_table =
{bg_str}

modified_gravity = yes
mu_function = constant
mu0 = {mu_const:.6f}
sigma_function = constant
sigma0 = 1.0

output = tCl pCl lCl mPk
lensing = yes
modes = s
l_max_scalars = 2500
P_k_max_h/Mpc = 10.0
k_max_tau0_over_l_max = 10.0
background_verbose = 0
verbose = 0
"""
with open("trc_final.ini", "w") as f:
    f.write(content)
print("Written: trc_final.ini")

# ----------------------------------------------------------------------
# 5. Run CLASS
# ----------------------------------------------------------------------
os.makedirs("output", exist_ok=True)
print("Running CLASS...")
result = subprocess.run(["./class_public/class", "trc_final.ini"], capture_output=True, text=True)
if result.returncode != 0:
    print("CLASS failed.")
    print(result.stderr)
    exit(1)
print("CLASS done.")

# ----------------------------------------------------------------------
# 6. Load TT spectrum
# ----------------------------------------------------------------------
tt_file = None
for pattern in ["trc_final*_cl_lensed.dat", "output/trc_final*_cl_lensed.dat"]:
    files = glob.glob(pattern)
    if files:
        tt_file = files[0]
        break
if tt_file is None:
    raise FileNotFoundError("TT file not found.")

data = np.loadtxt(tt_file)
ell_trc = data[:, 0]
tt_trc_dimless = data[:, 1]
factor = (2.7255 * 1e6)**2
tt_trc = tt_trc_dimless * factor

print(f"TT spectrum: {tt_file}")
print(f"Peak (μK²): {tt_trc.max():.1f}")

# ----------------------------------------------------------------------
# 7. Load Planck (if available)
# ----------------------------------------------------------------------
planck_file = None
for f in ["planck_tt.txt", "output/planck_tt.txt"]:
    if os.path.exists(f):
        planck_file = f
        break

if planck_file is None:
    print("Planck data not found. Skipping comparison.")
else:
    planck = np.loadtxt(planck_file, skiprows=1)
    ell_p = planck[:, 0]
    tt_p = planck[:, 1]
    err_p = planck[:, 2]

    # Interpolate
    mask = (ell_p >= max(ell_trc.min(), ell_p.min())) & (ell_p <= min(ell_trc.max(), ell_p.max()))
    ell_int = ell_p[mask]
    interp = interp1d(ell_trc, tt_trc, kind='cubic', fill_value='extrapolate')
    tt_trc_int = interp(ell_int)
    tt_p_int = tt_p[mask]
    err_int = err_p[mask]

    # χ²
    fit = (ell_int >= 30) & (ell_int <= 2500)
    chi2 = np.sum(((tt_trc_int[fit] - tt_p_int[fit]) / err_int[fit])**2)
    dof = np.sum(fit)
    red = chi2 / dof

    # Scaling
    def scale_chi2(A):
        return np.sum(((A * tt_trc_int[fit] - tt_p_int[fit]) / err_int[fit])**2)
    res = minimize_scalar(scale_chi2, bounds=(0.5,1.5), method='bounded')
    best_A = res.x
    red_scaled = res.fun / dof

    print(f"\n--- CMB TT vs Planck ---")
    print(f"χ²/dof (direct): {red:.3f}")
    print(f"Best scale factor: {best_A:.4f}")
    print(f"χ²/dof (scaled): {red_scaled:.3f}")

    # Plot
    plt.figure(figsize=(14,8))
    plt.subplot(2,1,1)
    plt.plot(ell_trc, tt_trc, 'b-', lw=2, label='TRC V16')
    plt.plot(ell_p, tt_p, 'r-', lw=1.2, alpha=0.8, label='Planck 2018')
    plt.errorbar(ell_p[::20], tt_p[::20], yerr=err_p[::20], fmt='ro', markersize=2, alpha=0.3)
    plt.xscale('log'); plt.yscale('log')
    plt.xlim(2,2500); plt.ylim(1e-2,1e4)
    plt.xlabel(r'$\ell$'); plt.ylabel(r'$D_\ell^{TT}$ [$\mu$K$^2$]')
    plt.title(f'TRC V16 vs Planck, χ²/dof = {red:.3f}')
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.subplot(2,1,2)
    residuals = (tt_trc_int - tt_p_int) / err_int
    plt.plot(ell_int, residuals, 'b-', lw=1)
    plt.axhline(0, color='black', linestyle='--')
    plt.axhline(2, color='gray', linestyle=':', alpha=0.6)
    plt.axhline(-2, color='gray', linestyle=':', alpha=0.6)
    plt.fill_between(ell_int, -2, 2, color='gray', alpha=0.1)
    plt.xscale('log'); plt.xlim(30,2500); plt.ylim(-5,5)
    plt.xlabel(r'$\ell$'); plt.ylabel('Residuals / σ')
    plt.title(f'Residuals, χ²/dof = {red:.3f}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ----------------------------------------------------------------------
# 8. Load and plot P(k)
# ----------------------------------------------------------------------
pk_file = None
for pattern in ["trc_final*_pk.dat", "output/trc_final*_pk.dat"]:
    files = glob.glob(pattern)
    if files:
        pk_file = files[0]
        break
if pk_file is not None:
    pk = np.loadtxt(pk_file)
    plt.figure(figsize=(10,6))
    plt.loglog(pk[:,0], pk[:,1], 'b-', lw=2)
    plt.xlabel(r'$k$ [h/Mpc]')
    plt.ylabel(r'$P(k)$ [(Mpc/h)$^3$]')
    plt.title('TRC V16 – Matter Power Spectrum at z=0')
    plt.grid(True, alpha=0.3)
    plt.xlim(1e-3, 10)
    plt.show()
else:
    print("P(k) file not found.")

# ----------------------------------------------------------------------
# 9. Extract τ, σ₈ from log
# ----------------------------------------------------------------------
log_file = None
for f in ["trc_final.log", "output/trc_final.log"]:
    if os.path.exists(f):
        log_file = f
        break

sigma8_known = 1.286  # from ODE calibration (fallback)
tau_known = 0.0544

if log_file is not None:
    with open(log_file, "r") as f:
        log = f.read()
        for line in log.split("\n"):
            if "tau_reio = " in line:
                tau_known = float(re.findall(r"tau_reio = ([\d.]+)", line)[0])
            if "sigma8 = " in line:
                sigma8_known = float(re.findall(r"sigma8 = ([\d.]+)", line)[0])
    print(f"\nFrom CLASS log: τ = {tau_known:.4f}, σ₈ = {sigma8_known:.4f}")
else:
    print("\nLog file not found; using fallback values.")

# ----------------------------------------------------------------------
# 10. Compute derived quantities
# ----------------------------------------------------------------------
Omega_m = TRC["Omega_m"]
S8 = sigma8_known * np.sqrt(Omega_m / 0.3)
A_lens = (sigma8_known / 0.811)**2 * (Omega_m / 0.315)**1.2

# ----------------------------------------------------------------------
# 11. Print final parameters
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("TRC V16 – Final Parameters")
print("="*60)
print(f"H0 = {TRC['H0']:.2f} km/s/Mpc")
print(f"Ω_m = {TRC['Omega_m']:.4f}")
print(f"Ω_b = {TRC['Omega_b']:.4f}")
print(f"β = {TRC['beta']:.3f}  → μ = {mu_const:.4f}")
print(f"τ_reio = {tau_known:.4f}")
print(f"σ₈ = {sigma8_known:.4f}")
print(f"S8 = σ₈ * sqrt(Ω_m/0.3) = {S8:.4f}")
print(f"A_lens = (σ₈/0.811)^2 * (Ω_m/0.315)^1.2 = {A_lens:.4f}")
print("="*60)

CLASS not found. Cloning and compiling...
